In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import copy
import random
import os
from tqdm import tqdm

# ==========================================
# 1. UTILS
# ==========================================
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

def get_device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_class_subset(dataset, classes):
    indices = [i for i, label in enumerate(dataset.targets) if label in classes]
    return Subset(dataset, indices)

def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.register(model)
    def register(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()
    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data = self.shadow[name]

def build_replay_buffer(dataset, classes, per_class=1000):
    indices = []
    counts = {k:0 for k in classes}
    for i, label in enumerate(dataset.targets):
        if label in classes and counts[label] < per_class:
            indices.append(i)
            counts[label] += 1
    return Subset(dataset, indices)

# ==========================================
# 2. ARCHITECTURE
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False), nn.BatchNorm2d(out_c))
    def forward(self, x): return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.final_conv = nn.Conv2d(256, 64, 1)
        
    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = []
        layers.append(ResidualBlock(in_c, out_c, stride))
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = self.final_conv(x)
        return x

class DynamicFusionModule(nn.Module):
    def __init__(self, in_channels, old_dim, new_dim, novelty_score=0.0):
        super().__init__()
        self.has_expansion = new_dim > 0
        self.novelty_score = novelty_score
        
        # Reuse Path: Kept Dropout for regularization
        self.proj_reuse = nn.Sequential(
            nn.Conv2d(in_channels, old_dim, 1, bias=False),
            nn.BatchNorm2d(old_dim),
            nn.ReLU(),
            nn.Dropout(0.2) 
        )
        
        # Expansion Path: REMOVED Dropout to let signal flow freely (Fix #2)
        if self.has_expansion:
            self.proj_expand = nn.Sequential(
                nn.Conv2d(in_channels, new_dim, 1, bias=False),
                nn.BatchNorm2d(new_dim),
                nn.ReLU()
            )
        
        self.gate_reuse = nn.Parameter(torch.tensor([0.0])) 

    def forward(self, x):
        # FIX #1: REUSE FLOOR (Max(0.15, 1-Novelty))
        # Ensure at least 15% signal comes from old brain
        reuse_scale = max(0.15, 1.0 - self.novelty_score)
        
        delta_old = self.proj_reuse(x) * torch.sigmoid(self.gate_reuse) * reuse_scale
        
        feat_new = None
        if self.has_expansion:
            feat_new = self.proj_expand(x)
            
        return delta_old, feat_new

class Broker(nn.Module):
    def __init__(self, n_specs, ch, n_classes, old_dim=64, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.projections = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(n_specs)])
        
        self.old_dim = old_dim
        self.new_dim = new_dim
        
        if new_dim > 0 or n_specs > 2:
             self.fusion = DynamicFusionModule(ch, old_dim, new_dim, novelty_score)
        else:
             self.fusion = None

        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(n_specs*ch, n_specs*ch//4, 1), nn.ReLU(), 
            nn.Conv2d(n_specs*ch//4, n_specs*ch, 1), nn.Sigmoid()
        )
        self.spatial_gate = nn.Sequential(nn.Conv2d(2, 1, 7, 1, 3), nn.Sigmoid())
        
        self.bottleneck = nn.Sequential(nn.Conv2d(n_specs*ch, old_dim, 1), nn.ReLU())
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(old_dim + new_dim, n_classes)

    def forward(self, feats):
        feats_resized = [F.interpolate(f, size=(4,4)) for f in feats]
        proj = []
        fusion_delta_old = None
        fusion_feat_new = None
        
        for i, (p, f) in enumerate(zip(self.projections, feats_resized)):
            if self.fusion is not None and i == len(feats) - 1:
                proj.append(p(f))
                fusion_delta_old, fusion_feat_new = self.fusion(f)
            else:
                proj.append(p(f))

        z_raw = torch.cat(proj, 1)
        w_c = self.channel_gate(z_raw)
        z = z_raw * w_c
        avg_out = torch.mean(z, 1, keepdim=True)
        max_out, _ = torch.max(z, 1, keepdim=True)
        w_s = self.spatial_gate(torch.cat([avg_out, max_out], 1))
        z = z * w_s
        
        base_memory = self.bottleneck(z)
        
        if fusion_delta_old is not None:
            enhanced_old = base_memory + fusion_delta_old
            if fusion_feat_new is not None:
                final_features = torch.cat([enhanced_old, fusion_feat_new], dim=1)
            else:
                final_features = enhanced_old
        else:
            final_features = base_memory

        logits = self.classifier(self.pool(final_features).flatten(1))
        
        return logits, base_memory, fusion_feat_new 

class KFN(nn.Module):
    def __init__(self, n_classes=5, n_specs=2, old_dim=64, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        self.broker = Broker(n_specs, 64, n_classes, old_dim, new_dim, novelty_score)
    def forward(self, x): return self.broker([s(x) for s in self.specialists])

# ==========================================
# 3. CORE LOGIC
# ==========================================

def compute_novelty_and_expand(model, specialist, loader, device, base_dim=64, threshold=0.25):
    print("\n[Phase 2.5] Measuring Novelty...")
    model.eval()
    specialist.eval()
    novelty_scores = []
    
    with torch.no_grad():
        for x, _ in tqdm(loader, desc="Scanning subspace", leave=False):
            x = x.to(device)
            f_spec = specialist(x).flatten(1)
            f_spec_norm = F.normalize(f_spec, p=2, dim=1)
            _, f_old, _ = model(x)
            f_old_flat = f_old.flatten(1)
            f_old_norm = F.normalize(f_old_flat, p=2, dim=1)
            dot = torch.sum(f_spec_norm * f_old_norm, dim=1, keepdim=True)
            projection = dot * f_old_norm
            residual = f_spec_norm - projection
            score = torch.norm(residual, p=2, dim=1).mean().item()
            novelty_scores.append(score)
            
    avg_novelty = np.mean(novelty_scores)
    print(f"--> Measured Novelty: {avg_novelty:.4f}")
    
    new_dim = 0
    if avg_novelty > threshold:
        # Boosted Expansion Cap: Allow up to 48 (was 32 + small buffer)
        raw_dim = int(np.ceil(base_dim * avg_novelty))
        dynamic_cap = min(int(48 + 0.1 * avg_novelty * base_dim), 64)
        new_dim = min(raw_dim, dynamic_cap)
        
        if raw_dim > dynamic_cap:
            print(f"--> Capping expansion at {dynamic_cap} (Raw was {raw_dim})")
        print(f"--> Novelty > {threshold}. Allocating {new_dim} NEW channels.")
    else:
        print(f"--> Novelty <= {threshold}. Standard Reuse.")
        new_dim = 4 
        print(f"--> (Safety Net) Allocating {new_dim} channels.")
        
    return new_dim, avg_novelty

def copy_layer(old_gate, num_old, num_new, ch_per_spec):
    old_in = num_old * ch_per_spec
    new_in = (num_old + num_new) * ch_per_spec
    old_hidden = old_in // 4
    new_hidden = new_in // 4
    new_conv1 = nn.Conv2d(new_in, new_hidden, 1)
    new_conv2 = nn.Conv2d(new_hidden, new_in, 1)
    with torch.no_grad():
        new_conv1.weight[:old_hidden, :old_in] = old_gate[1].weight
        new_conv1.bias[:old_hidden] = old_gate[1].bias
        new_conv2.weight[:old_in, :old_hidden] = old_gate[3].weight
        new_conv2.bias[:old_in] = old_gate[3].bias
    return nn.Sequential(nn.AdaptiveAvgPool2d(1), new_conv1, nn.ReLU(), new_conv2, nn.Sigmoid())

def expand_classifier(old_layer, new_num_classes, old_dim, new_dim):
    total_dim = old_dim + new_dim
    new_layer = nn.Linear(total_dim, new_num_classes)
    with torch.no_grad():
        new_layer.weight[:5, :old_dim] = old_layer.weight
        new_layer.bias[:5] = old_layer.bias
        nn.init.kaiming_normal_(new_layer.weight[5:, :], nonlinearity='relu')
        new_layer.bias[5:].zero_()
        if new_dim > 0:
            nn.init.kaiming_normal_(new_layer.weight[:5, old_dim:], nonlinearity='relu')
            new_layer.weight[:5, old_dim:] *= 0.01 
    return new_layer

def evaluate(model, loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _, _ = model(x)
            if isinstance(logits, tuple): logits = logits[0]
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

# ==========================================
# 4. TRAINING & MAIN
# ==========================================

def train_base_phase1(model, loader, test_loader, device):
    print("\n[Phase 1] Training Base Model (Task A)...")
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    ema = EMA(model, decay=0.999)
    
    for ep in range(100): 
        model.train()
        for x, y in tqdm(loader, desc=f"Ep {ep+1}", leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            inputs, ta, tb, lam = mixup_data(x, y, 0.4)
            logits, _, _ = model(inputs)
            loss = mixup_criterion(criterion, logits, ta, tb, lam)
            loss.backward()
            opt.step()
            ema.update(model)
        sched.step()
    
    ema.apply_shadow(model)
    acc = evaluate(model, test_loader, device)
    print(f"--> Phase 1 Accuracy: {acc:.2f}%")
    return model, acc

def train_specialist_phase2(loader, device):
    print("\n[Phase 2] Training Independent Specialist (Task B)...")
    spec = Specialist().to(device)
    head = nn.Linear(64, 5).to(device)
    opt = torch.optim.AdamW(list(spec.parameters()) + list(head.parameters()), lr=1e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    for ep in range(30):
        spec.train()
        for x, y in tqdm(loader, desc=f"Spec Ep {ep+1}", leave=False):
            x, y = x.to(device), (y - 5).to(device)
            opt.zero_grad()
            feats = spec(x)
            logits = head(F.adaptive_avg_pool2d(feats, 1).flatten(1))
            loss = criterion(logits, y)
            loss.backward()
            opt.step()
    print("--> Specialist Trained.")
    return spec

def integrate_phase3(model, specialist, teacher, loader_B, replay_loader, test_A, test_B, device, new_dim, novelty_score):
    print(f"\n[Phase 3] Dynamic Integration (Dim={new_dim}, Novelty={novelty_score:.3f})...")
    
    model.specialists.append(specialist)
    old_b = model.broker
    
    new_b = Broker(n_specs=3, ch=64, n_classes=10, old_dim=64, new_dim=new_dim, novelty_score=novelty_score).to(device)
    
    new_b.projections[0].load_state_dict(old_b.projections[0].state_dict())
    new_b.projections[1].load_state_dict(old_b.projections[1].state_dict())
    new_b.spatial_gate.load_state_dict(old_b.spatial_gate.state_dict())
    new_b.channel_gate = copy_layer(old_b.channel_gate, 2, 1, 64).to(device)
    
    with torch.no_grad():
        old_w = old_b.bottleneck[0].weight.data
        new_w = torch.zeros_like(new_b.bottleneck[0].weight.data)
        new_w[:, :128, :, :] = old_w 
        new_b.bottleneck[0].weight.data = new_w
        new_b.bottleneck[0].bias.data = old_b.bottleneck[0].bias.data.clone()
        
    new_b.classifier = expand_classifier(old_b.classifier, 10, 64, new_dim).to(device)
    
    model.broker = new_b
    
    for p in model.parameters(): p.requires_grad = False
    for p in model.broker.fusion.parameters(): p.requires_grad = True
    for p in model.broker.classifier.parameters(): p.requires_grad = True
    
    opt = torch.optim.AdamW([
        {'params': model.broker.fusion.parameters(), 'lr': 3e-3},
        {'params': model.broker.classifier.parameters(), 'lr': 1e-4}
    ])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    replay_iter = iter(replay_loader)
    
    LAMBDA_STABILITY = 25.0
    LAMBDA_ORTHO = 0.1 # FIX #3: Relaxed Orthogonality (Was 0.5)
    
    for ep in range(75):
        model.train()
        for x_B, y_B in tqdm(loader_B, desc=f"Integration Ep {ep+1}", leave=False):
            x_B, y_B = x_B.to(device), y_B.to(device)
            
            try: x_A, y_A = next(replay_iter)
            except: 
                replay_iter = iter(replay_loader)
                x_A, y_A = next(replay_iter)
            x_A, y_A = x_A.to(device), y_A.to(device)
            
            opt.zero_grad()
            
            logits_B, mem_B_old, mem_B_new = model(x_B)
            loss_B = criterion(logits_B, y_B)
            
            logits_A, mem_A_old, mem_A_new = model(x_A)
            loss_A = criterion(logits_A, y_A)
            
            with torch.no_grad():
                logits_T, teacher_mem_A, _ = teacher(x_A)
            loss_dist = F.kl_div(F.log_softmax(logits_A[:,:5]/2, 1), 
                                 F.softmax(logits_T[:,:5]/2, 1), reduction='batchmean')
            
            loss_stability = F.mse_loss(mem_A_old, teacher_mem_A)
            
            loss_ortho = torch.tensor(0.0).to(device)
            if new_dim > 0 and mem_B_new is not None:
                f_old = mem_B_old.flatten(1)
                f_new = mem_B_new.flatten(1)
                f_old = F.normalize(f_old, p=2, dim=0)
                f_new = F.normalize(f_new, p=2, dim=0)
                loss_ortho = torch.norm(torch.mm(f_old.T, f_new), p='fro')
            
            total_loss = loss_B + 1.5*loss_A + loss_dist + (LAMBDA_STABILITY * loss_stability) + (LAMBDA_ORTHO * loss_ortho)
            
            total_loss.backward()
            
            if model.broker.bottleneck[0].weight.grad is not None:
                model.broker.bottleneck[0].weight.grad.zero_()
                model.broker.bottleneck[0].bias.grad.zero_()
            
            opt.step()
            
    return model

def run():
    set_deterministic(42)
    device = get_device()
    os.makedirs("logs_balanced", exist_ok=True)
    
    stats = ((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    train_T = transforms.Compose([
        transforms.RandomCrop(32, 4), transforms.RandomHorizontalFlip(), 
        transforms.ToTensor(), transforms.Normalize(*stats)
    ])
    test_T = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])
    
    full_train = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_T)
    full_test = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_T)
    
    loader_A = DataLoader(get_class_subset(full_train, [0,1,2,3,4]), 128, shuffle=True, num_workers=2)
    loader_B = DataLoader(get_class_subset(full_train, [5,6,7,8,9]), 128, shuffle=True, num_workers=2)
    test_A = DataLoader(get_class_subset(full_test, [0,1,2,3,4]), 128, num_workers=2)
    test_B = DataLoader(get_class_subset(full_test, [5,6,7,8,9]), 128, num_workers=2)
    
    model = KFN(n_classes=5, n_specs=2, old_dim=64, new_dim=0).to(device)
    model, acc_A_init = train_base_phase1(model, loader_A, test_A, device)
    teacher = copy.deepcopy(model).eval()
    
    specialist = train_specialist_phase2(loader_B, device)
    
    subset_B = DataLoader(get_class_subset(full_test, [5,6,7,8,9]), 128, shuffle=False)
    new_dim, novelty_score = compute_novelty_and_expand(model, specialist, subset_B, device)
    
    replay_loader = DataLoader(build_replay_buffer(full_train, [0,1,2,3,4]), 64, shuffle=True)
    model = integrate_phase3(model, specialist, teacher, loader_B, replay_loader, test_A, test_B, device, new_dim, novelty_score)
    
    acc_A = evaluate(model, test_A, device)
    acc_B = evaluate(model, test_B, device)
    
    retention_pct = (acc_A / acc_A_init) * 100
    param_count = count_params(model)
    
    print("\n" + "="*40)
    print("FINAL RESULTS (BALANCED EDITION)")
    print("="*40)
    print(f"Total Params: {param_count/1e6:.2f}M")
    print(f"Novelty-Based Expansion: +{new_dim} Channels (Score: {novelty_score:.3f})")
    print("-" * 20)
    print(f"Task A (Initial): {acc_A_init:.2f}%")
    print(f"Task A (Final):   {acc_A:.2f}%")
    print(f"Task B (Final):   {acc_B:.2f}%")
    print(f"RETENTION RATE:   {retention_pct:.1f}%")
    print("="*40)

if __name__ == "__main__":
    run()

100%|██████████| 170M/170M [00:03<00:00, 45.7MB/s] 



[Phase 1] Training Base Model (Task A)...


--> Phase 1 Accuracy: 93.68%

[Phase 2] Training Independent Specialist (Task B)...


--> Specialist Trained.

[Phase 2.5] Measuring Novelty...


--> Measured Novelty: 0.9976
--> Capping expansion at 54 (Raw was 64)
--> Novelty > 0.25. Allocating 54 NEW channels.

[Phase 3] Dynamic Integration (Dim=54, Novelty=0.998)...



FINAL RESULTS (BALANCED EDITION)
Total Params: 8.43M
Novelty-Based Expansion: +54 Channels (Score: 0.998)
--------------------
Task A (Initial): 93.68%
Task A (Final):   75.66%
Task B (Final):   79.78%
RETENTION RATE:   80.8%
